<div style="padding:48px 42px;border:1px solid #d7e2ee;border-radius:10px;background:#f8fbff;">
<h1 style="font-size:2.3em;font-weight:800;margin:0 0 12px;">Hyperloom Workshop</h1>
<h2 style="font-size:1.24em;font-weight:500;color:#1b5e8f;margin:0 0 22px;">Agentic inference optimization for Qwen3-30B-A3B on AMD MI300X</h2>
<hr style="border:0;border-top:1px solid #d7e2ee;margin:18px 0;">
<p style="margin:0;color:#384454;font-size:1.02em;">Runtime: vLLM | Kernel backend: Forge | Workload: CONC=16, ISL=1024, OSL=1024 | Target gain: 10%</p>
</div>

## Workshop Roadmap

This notebook demonstrates how Hyperloom runs an evidence-driven optimization loop for an LLM serving workload. It uses one example run as the data source, then finishes with the exact notebook cells needed to launch and monitor a fresh run.

| # | Topic | What to look for |
|---|-------|------------------|
| 1 | Hyperloom architecture | How the coordinator, agents, profilers, and kernel backends fit together |
| 2 | Optimization loop | Baseline -> profile -> tune -> validate -> report |
| 3 | Example run | Qwen3-30B-A3B on vLLM, MI300X, CONC=16 |
| 4 | Performance gain | Baseline, best variant, validated gain, and target status |
| 5 | Why the gain happens | Memory-bound MoE/GEMM bottleneck and Forge-selected vLLM tuned configs |
| 6 | Live run cells | Startup script cell and monitor cell |

The notebook intentionally treats terminal logs as secondary. The run state, benchmark artifacts, and final report are the source of truth.

---
## 1. Hyperloom Architecture

Hyperloom treats inference optimization as a measured search problem. The user provides a model, runtime, hardware target, and workload. Hyperloom then coordinates agents and tools that repeatedly propose one change, benchmark it, and keep only validated wins.

![Hyperloom loop](slides/hyperloom_loop.png)

![Simplified Hyperloom architecture](slides/simplified_fig1.png)

| Layer | Role in the workflow |
|-------|----------------------|
| `inference_optimizer` / Coordinator | Starts the run, creates the session, owns phase sequencing, action execution, policy gates, and final reporting. |
| Orchestration agent | Plans the next allowed action and adapts from benchmark results. |
| Kernel agent | Routes hot kernels into TraceLens analysis and kernel optimization backends. |
| Critic agent | Reviews risky source or kernel changes before they are promoted. |
| Robustness agent | Watches stalls, crashes, process health, and recovery signals. |
| Framework agent | Discovers relevant vLLM/SGLang framework-source candidates when that phase is enabled. |
| Magpie / InferenceX | Launches serving benchmarks and records throughput evidence for vLLM/SGLang/Atom workloads. |
| TraceLens / roofline model | Converts traces and model metadata into bottleneck direction, kernel candidates, and performance ceilings. |
| Forge / GEAK / OOB backends | Tune or rewrite kernels, then return a candidate that still must pass end-to-end validation. |

The important design point is that Hyperloom does not accept a microbenchmark result by itself. A candidate becomes the current best only after it improves the actual serving workload under the same measurement path.

---
## 2. Optimization Loop

The live runtime phase chain is:

```text
PRELUDE -> FRAMEWORK_PR -> EXPLORE -> KERNEL -> SWEEP -> CLOSE
```

| Phase | Purpose |
|-------|---------|
| PRELUDE | Establish target comparison, measure the baseline, and collect the first profile or roofline signal. |
| FRAMEWORK_PR | Optionally test framework-source candidates discovered by the framework agent. |
| EXPLORE | Search serving parameters, environment knobs, and source-patch ideas. |
| KERNEL | Route hot kernels from TraceLens into Forge, GEAK, or other kernel backends. |
| SWEEP | Check that the optimized stack still wins across workload frontiers. |
| CLOSE | Write the final report and machine-readable run summary. |

A full session records `manifest.json`, `state.json`, per-action `runs/`, agent workspaces, reports, and, when closeout completes normally, `session_breakdown.json`. For dashboards, `session_breakdown.json` is the stable external contract. This example uses `state.json` and raw benchmark artifacts because the run ended at the wall-time limit and the safety-net report path was used.

---
## 3. Example Run Configuration

The example run uses a vLLM serving workload for Qwen3-30B-A3B:

| Item | Value |
|------|-------|
| Model | `Qwen-Qwen3-30B-A3B` |
| Framework | `vllm` |
| GPU type | `mi300x` |
| Tensor parallelism | `TP=1` |
| Concurrency | `CONC=16` |
| Prompt / output length | `ISL=1024`, `OSL=1024` |
| Precision | `bf16` |
| Target | `10%` throughput gain |
| Time budget | `1 hour` |
| Winning backend | `Forge` |

Set `HYPERLOOM_EXAMPLE_SESSION_DIR` if the example artifacts are stored outside the standard Hyperloom workspace roots.

In [ ]:
from pathlib import Path
import json
import os
from pprint import pprint


EMBEDDED_EXAMPLE_SUMMARY = {
    "framework": "vllm",
    "concurrency": 16,
    "sequence_lengths": {"isl": 1024, "osl": 1024},
    "baseline_tok_s_per_gpu": 1375.208,
    "hot_baseline_tok_s_per_gpu": 1441.083,
    "best_tok_s_per_gpu": 1454.510,
    "validated_gain_pct": 5.7665,
    "gain_vs_hot_baseline_pct": 0.9317,
    "best_action": "gemm_tuning",
    "best_backend": "forge",
    "best_variant": "vllm_moe_triton",
    "stop_reason": "example_run_complete",
    "roofline_bound": "memory",
    "within_roofline_pct": 68.5,
    "gap_to_roofline_pct": 31.5,
    "top_ops": [
        {
            "name": "moe_fused",
            "pct_time": 88.41,
            "bound": "memory",
            "arithmetic_intensity": 1.55,
        },
        {
            "name": "attention_decode",
            "pct_time": 5.62,
            "bound": "memory",
            "arithmetic_intensity": 0.92,
        },
        {
            "name": "sampling_and_logits",
            "pct_time": 2.48,
            "bound": "memory",
            "arithmetic_intensity": 0.76,
        },
    ],
    "report_exists": False,
    "current_setting_exists": False,
    "session_breakdown_exists": False,
    "data_source": "embedded_example",
}


def find_example_session():
    override = os.environ.get("HYPERLOOM_EXAMPLE_SESSION_DIR")
    if override:
        path = Path(override)
        if not (path / "state.json").exists():
            raise FileNotFoundError(f"HYPERLOOM_EXAMPLE_SESSION_DIR has no state.json: {path}")
        return path

    patterns = [
        "/workspace/hyperloom*/Qwen-Qwen3-30B-A3B/*/state.json",
        "/workspace/hyperloom*/Qwen3-30B-A3B/*/state.json",
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(Path("/").glob(pattern.lstrip("/")))
    candidates = sorted(candidates, key=lambda path: path.stat().st_mtime, reverse=True)
    if not candidates:
        return None
    return candidates[0].parent


EXAMPLE_SESSION_DIR = find_example_session()

if EXAMPLE_SESSION_DIR is None:
    summary = dict(EMBEDDED_EXAMPLE_SUMMARY)
else:
    STATE_PATH = EXAMPLE_SESSION_DIR / "state.json"
    REPORT_PATH = EXAMPLE_SESSION_DIR / "reports" / "final.md"
    CURRENT_SETTING_PATH = EXAMPLE_SESSION_DIR / "current_setting.sh"
    SESSION_BREAKDOWN_PATH = EXAMPLE_SESSION_DIR / "session_breakdown.json"

    state = json.loads(STATE_PATH.read_text())
    current_best = state.get("current_best") or {}
    roofline = state.get("baseline_roofline_ceiling") or {}
    perfmodel = roofline.get("perfmodel_breakdown") or {}
    ops = sorted(perfmodel.get("ops") or [], key=lambda op: op.get("pct_time", 0), reverse=True)

    baseline = float(state.get("baseline_tput") or 0.0)
    hot_baseline = float(state.get("baseline_hot_tput") or 0.0)
    best = float(current_best.get("tput") or 0.0)
    gain = ((best / baseline) - 1.0) * 100.0 if baseline else None
    hot_gain = ((best / hot_baseline) - 1.0) * 100.0 if hot_baseline else None

    summary = {
        "framework": state.get("framework"),
        "concurrency": state.get("conc"),
        "sequence_lengths": {"isl": state.get("isl"), "osl": state.get("osl")},
        "baseline_tok_s_per_gpu": baseline,
        "hot_baseline_tok_s_per_gpu": hot_baseline,
        "best_tok_s_per_gpu": best,
        "validated_gain_pct": state.get("cumulative_gain_validated", gain),
        "gain_vs_hot_baseline_pct": hot_gain,
        "best_action": current_best.get("action"),
        "best_backend": current_best.get("engine"),
        "best_variant": current_best.get("variant_name"),
        "stop_reason": state.get("stop_reason"),
        "roofline_bound": roofline.get("roofline_bound_kind"),
        "within_roofline_pct": roofline.get("within_roofline_pct"),
        "gap_to_roofline_pct": roofline.get("gap_to_roofline_pct"),
        "top_ops": [
            {
                "name": op.get("name"),
                "pct_time": round(float(op.get("pct_time", 0)) * 100, 2),
                "bound": op.get("bound"),
                "arithmetic_intensity": round(float(op.get("ai", 0)), 2),
            }
            for op in ops[:5]
        ],
        "report_exists": REPORT_PATH.exists(),
        "current_setting_exists": CURRENT_SETTING_PATH.exists(),
        "session_breakdown_exists": SESSION_BREAKDOWN_PATH.exists(),
        "data_source": str(EXAMPLE_SESSION_DIR),
    }

pprint(summary)

---
## 4. Performance Gain

The example run reached a validated 5%+ throughput gain.

<div style="display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:16px;margin:14px 0 18px;">
<div style="background:#f6f8ff;border:1px solid #dce5ff;border-radius:8px;padding:18px;"><div style="font-size:1.45em;font-weight:800;color:#2b6cb0;">1375.2</div><div style="font-weight:700;color:#202938;margin-top:4px;">baseline tok/s/GPU</div><div style="color:#5b6678;margin-top:6px;">gain accounting anchor</div></div>
<div style="background:#f4fff7;border:1px solid #ccebd6;border-radius:8px;padding:18px;"><div style="font-size:1.45em;font-weight:800;color:#2f855a;">1454.5</div><div style="font-weight:700;color:#202938;margin-top:4px;">best tok/s/GPU</div><div style="color:#5b6678;margin-top:6px;">Forge GEMM tuned</div></div>
<div style="background:#fff8ef;border:1px solid #f7d7ad;border-radius:8px;padding:18px;"><div style="font-size:1.45em;font-weight:800;color:#c05621;">+5.77%</div><div style="font-weight:700;color:#202938;margin-top:4px;">validated gain</div><div style="color:#5b6678;margin-top:6px;">5%+ gain reached</div></div>
</div>

The winning action was `gemm_tuning` with the `forge` backend and `vllm_moe_triton` tuner. Hyperloom promoted it because the tuned vLLM configuration improved the same end-to-end serving benchmark used for gain accounting.

A stricter comparison against the later hot baseline measurement, `1441.1 tok/s/GPU`, gives a smaller `+0.93%` delta. The notebook reports Hyperloom's validated run-state gain because that is the value used by the optimizer's promotion and target accounting.

In [ ]:
baseline = float(summary["baseline_tok_s_per_gpu"])
hot_baseline = float(summary["hot_baseline_tok_s_per_gpu"] or 0)
best = float(summary["best_tok_s_per_gpu"])
gain_pct = float(summary["validated_gain_pct"])
top_ops = summary["top_ops"]

try:
    import matplotlib.pyplot as plt

    labels = ["Baseline", "Hot baseline", "Best"] if hot_baseline else ["Baseline", "Best"]
    values = [baseline, hot_baseline, best] if hot_baseline else [baseline, best]
    colors = ["#2b6cb0", "#718096", "#2f855a"] if hot_baseline else ["#2b6cb0", "#2f855a"]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].bar(labels, values, color=colors)
    axes[0].set_ylabel("tok/s/GPU")
    axes[0].set_title(f"End-to-end throughput (+{gain_pct:.2f}% run-state gain)")
    for i, value in enumerate(values):
        axes[0].text(i, value, f"{value:.1f}", ha="center", va="bottom")

    names = [op["name"] for op in top_ops]
    times = [op["pct_time"] for op in top_ops]
    axes[1].barh(names[::-1], times[::-1], color="#805ad5")
    axes[1].set_xlabel("Estimated decode time share (%)")
    axes[1].set_title("Top modeled bottlenecks")

    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(f"matplotlib chart skipped: {exc}")
    print(f"Baseline: {baseline:.2f} tok/s/GPU")
    if hot_baseline:
        print(f"Hot baseline: {hot_baseline:.2f} tok/s/GPU")
    print(f"Best: {best:.2f} tok/s/GPU")
    print(f"Gain: {gain_pct:.2f}%")
    print("Top ops:")
    for op in top_ops:
        print(f"  {op['name']}: {op['pct_time']:.2f}% time, bound={op['bound']}, AI={op['arithmetic_intensity']}")

---
## 5. Why This Run Gets Faster

The roofline estimate classifies the baseline as **memory-bound**. At the baseline anchor, the workload reaches about **68.5%** of the modeled memory roofline, leaving a **31.5%** gap to the estimated ceiling. The compute ceiling is far above the memory ceiling, so better data movement and kernel selection matter more than simply chasing more FLOPs.

The modeled decode breakdown points to one dominant target:

| Operation | Time share | Bound | Why it matters |
|-----------|------------|-------|----------------|
| `moe_fused` | `88.41%` | memory | Dominates decode time and moves about `37.34 GB` in the model estimate. |
| `sdpa` | `5.75%` | memory | Secondary attention cost. |
| `q_proj` / `o_proj` | `1.93%` each | memory | Projection work, much smaller than MoE. |
| `lm_head` | `1.49%` | memory | Output projection cost. |

Trace evidence points in the same direction: MoE GEMM kernels are among the top GPU consumers, including `ck::kernel_moe_gemm` entries at roughly `22.19%` and `12.23%` GPU time. That makes `vllm_moe_triton` GEMM tuning the natural first high-leverage action.

Forge tested `10` M-buckets for the vLLM MoE Triton path. The tuning artifact reports `best_micro_speedup=1.2976` and `avg_micro_speedup=1.0796`; the largest bucket-level win was at `M=48`, improving from `475.50 us` to `366.43 us`. The retained tuned config targets the measured hardware/model shape:

```text
E=128, N=768, device=AMD Instinct MI300X, dtype=bfloat16
```

The end-to-end gain is smaller than the best microbenchmark speedup because only part of serving time is affected, the workload remains memory-bound, and other runtime overheads stay unchanged. The result is still useful: Hyperloom identified the dominant MoE/GEMM path, generated a tuned vLLM config for that path, and validated a serving-throughput gain under the same workload.

---
## 6. Reproducibility Artifacts

A Hyperloom session leaves behind artifacts that make the result auditable:

| Artifact | Purpose |
|----------|---------|
| `state.json` | Current phase, baseline, current best, gain, stop reason, and roofline summary. |
| `manifest.json` | Run identity, workload, framework, and session metadata. |
| `session_breakdown.json` | Stable dashboard/reporting contract when CLOSE completes normally. |
| `runs/` | Per-action workspaces for baseline, profiling, tuning, sweep, and validation. |
| `reports/final.md` | Human-readable closeout report. |
| `current_setting.sh` | Generated launch recipe for the best known setting. |

For this example, use `state.json` plus the raw throughput JSONs as the metric source of truth. The final report was produced by the safety-net closeout path because the run hit the time budget. The throughput benchmark files were written and parsed, while the downstream accuracy eval path exited nonzero after throughput collection, so present this example as **throughput evidence**, not as a complete accuracy-validation report.

The key integration artifact is the `VLLM_TUNED_CONFIG_FOLDER` recorded in `current_best.extra_envs`. It points at the tuned vLLM MoE Triton configuration selected by Forge.

---
## 7. Live Demo

This section provides a way to run the same Hyperloom workshop workload. 

| Option | Best for | How it runs |
|--------|----------|-------------|
| **Claude-guided terminal run** | Interactive demos, visible conversation history, user-driven inspection, and DIY parameter changes. | Start Claude CLI with `--dangerously-skip-permissions`, paste the prepared prompt, and let the terminal session launch and monitor the run. |


Make sure no vLLM, SGLang, Magpie, or Hyperloom optimizer process is already active for the same GPUs. Both paths include a process guard, but checking first makes the live demo easier to explain.


### Claude-Guided Terminal Run

Ask Claude to inspect logs, explain artifacts, stop the run, or change parameters. 

Start Claude from a terminal in the project root. Use this only in a prepared workshop workspace where the CLI may operate without permission prompts:

```bash
cd /workspace/projects/Hyperloom
 
for f in /workspace/projects/Hyperloom/.env /workspace/Workshop_10_Hyperloom/.env; do
  if [ -f "$f" ]; then
    cp "$f" "$f.before-claude-auth-fix"
    sed -i '/ANTHROPIC_CUSTOM_HEADERS/d' "$f"
  fi
done
 
set -a
source /workspace/projects/Hyperloom/.env
set +a
 
unset ANTHROPIC_CUSTOM_HEADERS
 
export USER_DATA_PATH="${USER_DATA_PATH:-/workspace/hyperloom}"
export NPM_CONFIG_PREFIX="${NPM_CONFIG_PREFIX:-$USER_DATA_PATH/runtime/npm-global}"
export PATH="$NPM_CONFIG_PREFIX/bin:/workspace/hyperloom/runtime/py312-rocm/bin:/opt/rocm/bin:$PATH"
 
export ANTHROPIC_MODEL="MiniMax-M3[1m]"
export ANTHROPIC_DEFAULT_OPUS_MODEL="MiniMax-M3[1m]"
export ANTHROPIC_DEFAULT_SONNET_MODEL="MiniMax-M3[1m]"
export ANTHROPIC_DEFAULT_HAIKU_MODEL="MiniMax-M3[1m]"
export CLAUDE_CODE_SUBAGENT_MODEL="MiniMax-M3[1m]"
 
claude -p 'Reply with exactly: OK'

IS_SANDBOX=1 claude --dangerously-skip-permissions \
  --add-dir /workspace/projects/Hyperloom \
  --add-dir "$USER_DATA_PATH" \
  --name hyperloom-workshop
```

Paste this prompt into Claude:

```text
You are running a Hyperloom workshop live demo.

Run path:
- Use the Claude-guided terminal method for this demo.
- Do not run the direct notebook startup cells in the same session.
- Use the short presentation path with cold-start inheritance.
- Keep the demo focused on CONC=16 first.

Goal:
- Demonstrate a Hyperloom optimization run for Qwen3-30B-A3B on vLLM.
- Use CONC=16, ISL=1024, OSL=1024, TP=1, bf16, Forge backend, target gain 10, max hours 0.5.
- Start from inherited cold-start facts so the audience can see KERNEL/GEMM behavior quickly.

Cold-start inheritance:
- This run should create a fresh new Hyperloom session.
- It should inherit the previous cold-start baseline, warm replay, profile, roofline, and TraceLens facts from:
  /workspace/hyperloom/Qwen-Qwen3-30B-A3B/20260629T095828Z
- This is not resume.
- Do not rerun PRELUDE, FRAMEWORK_PR, EXPLORE, baseline, warm replay, profile, roofline, or TraceLens.
- The new session should start at KERNEL.

Operating rules:
- Work under /workspace/projects/Hyperloom unless the user explicitly redirects you.
- Do not print API keys, token values, .env contents, or secret-bearing config files.
- Before launching, check for active vLLM, SGLang, Magpie, VLLM::EngineCore, or inference_optimizer processes and report any conflict.
- If any conflicting serving or optimizer process exists, do not launch. Report the process list and ask whether to stop it.
- Prefer the writable Python runtime: /workspace/hyperloom/runtime/py312-rocm/bin/python.
- Keep npm global installs under /workspace/hyperloom/runtime/npm-global when npm setup is needed.
- Use the prepared image/runtime; do not run bootstrap.
- Do not make git commits or modify source files unless the user explicitly asks.
- If the user asks to stop the run, terminate the optimizer process group and verify no serving or optimizer process remains.

Environment setup before any check or launch:
- Run these commands without printing secrets:

  cd /workspace/projects/Hyperloom

  source /etc/profile.d/hyperloom-workshop.sh 2>/dev/null || true
  source /workspace/hyperloom/runtime/local-setup.env.sh
  source /workspace/hyperloom/runtime/kernel-agent.env.sh

  set -a
  source ./.env
  set +a

  unset ANTHROPIC_CUSTOM_HEADERS

  export USER_DATA_PATH="${USER_DATA_PATH:-/workspace/hyperloom}"
  export NPM_CONFIG_PREFIX="${NPM_CONFIG_PREFIX:-$USER_DATA_PATH/runtime/npm-global}"
  export PATH="$NPM_CONFIG_PREFIX/bin:/workspace/hyperloom/runtime/py312-rocm/bin:/opt/rocm/bin:$PATH"

  export PYTHON=/workspace/hyperloom/runtime/py312-rocm/bin/python
  export MAGPIE_PYTHON=/workspace/hyperloom/runtime/py312-rocm/bin/python
  export ROCM_PATH="${ROCM_PATH:-/opt/rocm}"
  export HIP_PATH="${HIP_PATH:-/opt/rocm}"
  export LD_LIBRARY_PATH="/opt/rocm/lib:/opt/rocm/lib64:/opt/python/lib/python3.13/site-packages/_rocm_sdk_devel/lib:${LD_LIBRARY_PATH:-}"
  export LIBRARY_PATH="/opt/rocm/lib:/opt/rocm/lib64:/opt/python/lib/python3.13/site-packages/_rocm_sdk_devel/lib:${LIBRARY_PATH:-}"

Suggested workflow:
1. Summarize the workload and confirm that this is the inherited CONC=16-focused demo.
2. Run preflight checks:
   - no conflicting vLLM, SGLang, Magpie, VLLM::EngineCore, or inference_optimizer processes
   - GPU occupancy and VRAM usage
   - model path: /workspace/models/Qwen-Qwen3-30B-A3B
   - inherited source: /workspace/hyperloom/Qwen-Qwen3-30B-A3B/20260629T095828Z
   - writable Python runtime
   - torch device_count and torch._C._cuda_getDeviceCount
   - vLLM import path
   - Forge tuner availability
   - Claude/MiniMax smoke test if needed
3. Launch Hyperloom in the background with this command:

   /workspace/hyperloom/runtime/py312-rocm/bin/python -m inference_optimizer.cli --verbose optimize \
     --orchestration-backend openai-tools \
     --claude-model MiniMax-M3 \
     --codex-model MiniMax-M3 \
     --model /workspace/models/Qwen-Qwen3-30B-A3B \
     --framework vllm --gpu-type mi300x --tp 1 --conc 16 \
     --isl 1024 --osl 1024 --max-model-len 6144 --precision bf16 \
     --target-gain 10 --max-hours 0.5 --tick-interval-sec 30 \
     --model-class moe_swa --gpu-specialist-capacity 0 \
     --no-warm-replay \
     --inherit-cold-start-from /workspace/hyperloom/Qwen-Qwen3-30B-A3B/20260629T095828Z

4. Save the PID, log path, launch-info path, and session directory under /workspace/hyperloom/optimizer_runs.
5. Monitor every 60 seconds:
   - process status
   - latest log tail
   - session_dir
   - state.json phase
   - state.json tick
   - baseline_tput
   - inherited current_best
   - cumulative_gain_validated
   - current best action and throughput
   - Forge GEMM warmup/measure status
   - any errors mentioning fused_experts, NotImplementedError, Invalid API key, ANTHROPIC_CUSTOM_HEADERS, or cannot find -lamdhip64
6. If Hyperloom tries to enter a long SWEEP or conc_sweep, explain the expected runtime and ask the user before continuing long sweeps.
7. Explain observed performance in terms of inherited baseline, inherited warm replay, KERNEL/GEMM action, and CONC=16 throughput.

Start by inspecting the project and current runtime state, then proceed with the launch only if the process guard is clear.
```

To resume the latest conversation from the same project directory:

```bash
claude --continue --dangerously-skip-permissions
```
